# 企业级数据 Pipeline 设计与实施手册
## GitHub Events → BigQuery (GCP Medallion Architecture)

> 本手册是**可执行的实施指南**。代码均对应 `/workspace` 目录下的真实实现，按章节顺序操作即可完成完整部署。

## 项目信息
| 项 | 值 |
|----|---------|
| GCP Project | `gcp-lab-data-engineering` |
| Region | `us-central1` |
| 数据源 | GitHub Public Events API |
| 调度频率 | 每日 (可改为实时 Pub/Sub) |
| 数据架构 | Medallion: raw → staging → serving |

## 章节索引
| # | 章节 | 操作类型 |
|---|------|----------|
| 0 | GCP 环境准备 | **执行** |
| 1 | 架构设计 | 阅读 |
| 2 | 基础设施 (Terraform) | **执行** |
| 3 | 数据摄取层 | 阅读 + **执行** |
| 4 | 数据质量框架 | 阅读 + **执行** |
| 5 | ETL 转换层 | 阅读 + **执行** |
| 6 | 编排与调度 | **执行** |
| 7 | 监控与可观测性 | 阅读 + **执行** |
| 8 | 安全与数据治理 | 阅读 |
| 9 | CI/CD | 阅读 |
| 10 | 实施检查清单 | **核对** |

---
# Part 0: GCP 环境准备

## 前置条件
1. 已创建 GCP Project (`gcp-lab-data-engineering`)
2. 本地已安装 `gcloud` CLI 并通过认证
3. 已安装 Terraform >= 1.3
4. Python >= 3.11

## 认证方式
```bash
# 方式 A: 本地开发 (ADC — Application Default Credentials)
gcloud auth application-default login

# 方式 B: CI/CD (Service Account Key)
export GOOGLE_APPLICATION_CREDENTIALS=/path/to/key.json

# 方式 C: Cloud Run / Compute Engine (自动继承 SA 权限，推荐生产)
# 无需任何配置
```

In [7]:
# ===== 0.1 项目配置 — 所有后续步骤共用 =====

PROJECT_ID  = "gcp-lab-data-engineering"   # <-- 修改为你的 project ID
REGION      = "us-central1"
RAW_BUCKET  = f"{PROJECT_ID}-datalake-raw"
STG_BUCKET  = f"{PROJECT_ID}-datalake-staging"
TEMP_BUCKET = f"{PROJECT_ID}-pipeline-temp"

# 数据层对应 BigQuery Dataset
BQ_LAYERS = {
    "raw":     f"{PROJECT_ID}.raw",
    "staging": f"{PROJECT_ID}.staging",
    "serving": f"{PROJECT_ID}.serving",
}

print(f"Project:   {PROJECT_ID}")
print(f"Region:    {REGION}")
print(f"GCS raw:   gs://{RAW_BUCKET}")
print(f"BQ raw:    {BQ_LAYERS['raw']}")
print(f"BQ stg:    {BQ_LAYERS['staging']}")
print(f"BQ serve:  {BQ_LAYERS['serving']}")

Project:   gcp-lab-data-engineering
Region:    us-central1
GCS raw:   gs://gcp-lab-data-engineering-datalake-raw
BQ raw:    gcp-lab-data-engineering.raw
BQ stg:    gcp-lab-data-engineering.staging
BQ serve:  gcp-lab-data-engineering.serving


In [3]:
# ===== 0.2 验证 GCP 认证 =====
import subprocess  #在 Python 里执行 shell 命令的包

# 定义一个工具函数 run(cmd)
# 用 subprocess 在 Python 里执行 shell 命令，返回命令的输出（stdout）。如果命令出错，打印 stderr。
def run(cmd: str) -> str:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                            env={**__import__('os').environ, 
                                 "PATH": "/home/vscode/google-cloud-sdk/bin:" + __import__('os').environ["PATH"]})
    if result.returncode != 0:
        print(f"ERROR: {result.stderr.strip()}")
    return result.stdout.strip()

print("当前账号:", run("gcloud auth list --filter=status=ACTIVE --format='value(account)'"))
print("当前项目:", run("gcloud config get-value project"))

当前账号: stefanzy0805@gmail.com
当前项目: gcp-lab-data-engineering


In [5]:
# ===== 0.3 安装 Python 依赖 =====
# 在 terminal 中执行:
# pip install -r /workspace/requirements.txt

# 或直接运行:
import subprocess
result = subprocess.run(
    "pip install -r /workspace/requirements.txt -q",
    shell=True, capture_output=True, text=True
)
print(result.stdout or "Dependencies installed.")
if result.stderr:
    print("STDERR:", result.stderr[-500:])

Dependencies installed.
STDERR: /vscode/.cache/pip' or its parent directory is not owned or is not writable by the current user. The cache has been disabled. Check the permissions and owner of that directory. If executing pip with sudo, you should use sudo's -H flag.

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip



---
# Part 1: 架构设计

## 1.1 整体数据流

```
GitHub Events API
       │  HTTPS polling (每日/实时)
       ▼
  GCS Raw Bucket                    ← ingestion/api_ingest.py
  gs://{project}-datalake-raw/
  raw/github_events/dt=YYYY-MM-DD/*.jsonl
       │  BigQuery Load Job
       ▼
  BigQuery: raw.github_events       ← etl/load.py
  (partitioned by _ingested_at, 原始 JSON 字段)
       │  SQL MERGE (去重 + 标准化)
       ▼
  BigQuery: staging.github_events   ← etl/transform.py
  (partitioned by event_date, clustered by event_type+repo_name)
       │  SQL MERGE (聚合)
       ▼
  BigQuery: serving.daily_event_stats   ← etl/transform.py
  BigQuery: serving.top_repos_daily
       │  直接查询 / BI 工具
       ▼
  Looker Studio / Tableau / API
```

## 1.2 GCP 服务映射

| 需求 | GCP 服务 | 对应目录/文件 |
|------|----------|---------------|
| 原始存储 | Cloud Storage (GCS) | `infrastructure/terraform/gcs.tf` |
| 数据仓库 | BigQuery | `infrastructure/terraform/bigquery*.tf` |
| 流式消息 | Cloud Pub/Sub | `infrastructure/terraform/pubsub.tf` |
| 调度 | Cloud Scheduler + Cloud Run | (Part 6) |
| 监控 | Cloud Monitoring | (Part 7) |
| 日志 | Cloud Logging | (Part 7) |
| 密钥 | Secret Manager | (Part 8) |
| 权限 | Cloud IAM | `infrastructure/terraform/iam.tf` |
| CI/CD | Cloud Build | (Part 9) |

## 1.3 数据分层规范

| 层 | BQ Dataset | GCS Bucket | 特征 | 保留 |
|----|-----------|------------|------|------|
| Raw (Bronze) | `raw` | `*-datalake-raw` | 原始 JSONL，只追加，不改字段 | 永久 （因为Raw 是唯一的"事实来源"） |
| Staging (Silver) | `staging` | `*-datalake-staging` | 清洗去重，标准字段，分区+聚簇 | 2 年 |
| Serving (Gold) | `serving` | — | 聚合宽表，BI 直接查询 | 5 年 |

## 1.4 幂等性保障

```
Idempotency means: no matter how many times you run the same operation, the result is always the same.
Raw:     GCS 对象名含时间戳 → 每次运行产生新文件，不覆盖
         BQ load job WRITE_APPEND → 重跑会写入重复数据
         → 解决: BQ load 前先删除当日分区 (partition overwrite)

Staging: SQL MERGE (WHEN NOT MATCHED THEN INSERT)
         → 重跑安全，已存在 event_id 不重复插入

Serving: SQL MERGE (WHEN MATCHED UPDATE / WHEN NOT MATCHED INSERT)
         → 重跑安全，聚合值会被更新为正确值
```

---
# Part 2: 基础设施 (Terraform)

## 目录结构
```
infrastructure/terraform/
├── providers.tf        ← GCP provider 配置
├── variables.tf        ← 输入变量
├── terraform.tfvars    ← 实际变量值 (project_id)
├── bigquery.tf         ← BigQuery datasets (raw/staging/serving)
├── bigquery_tables.tf  ← BigQuery tables + schemas
├── gcs.tf              ← GCS buckets + lifecycle rules
├── pubsub.tf           ← Pub/Sub topics + subscriptions
└── iam.tf              ← Service accounts + IAM bindings
```

## 已部署资源 (terraform.tfstate)
- ✅ BigQuery datasets: `raw`, `staging`, `serving`
- ✅ Service accounts: `svc-ingest`, `svc-etl`, `svc-analytics`
- ✅ IAM bindings

## 待部署资源
- ⬜ GCS buckets (`gcs.tf`)
- ⬜ Pub/Sub topics (`pubsub.tf`)
- ⬜ BigQuery tables with schemas (`bigquery_tables.tf`)

In [18]:
# ===== 2.1 查看 Terraform 计划 (不执行) =====
import subprocess

result = subprocess.run(
    f"cd /workspace/infrastructure/terraform && terraform plan -var='project_id={PROJECT_ID}' -var='region={REGION}' 2>&1 | tail -30",
    shell=True, capture_output=True, text=True
)
print(result.stdout)

      + etag   = (known after apply)
      + id     = (known after apply)
      + member = "serviceAccount:svc-etl@gcp-lab-data-engineering.iam.gserviceaccount.com"
      + role   = "roles/storage.objectAdmin"
    }

  # google_storage_bucket_iam_member.etl_write_temp will be created
  + resource "google_storage_bucket_iam_member" "etl_write_temp" {
      + bucket = "gcp-lab-data-engineering-pipeline-temp"
      + etag   = (known after apply)
      + id     = (known after apply)
      + member = "serviceAccount:svc-etl@gcp-lab-data-engineering.iam.gserviceaccount.com"
      + role   = "roles/storage.objectAdmin"
    }

  # google_storage_bucket_iam_member.ingest_write_raw will be created
  + resource "google_storage_bucket_iam_member" "ingest_write_raw" {
      + bucket = "gcp-lab-data-engineering-datalake-raw"
      + etag   = (known after apply)
      + id     = (known after apply)
      + member = "serviceAccount:svc-ingest@gcp-lab-data-engineering.iam.gserviceaccount.com"
      + r

In [7]:
# ===== 2.2 应用 Terraform (实际部署基础设施) =====
# ⚠️ 这会在 GCP 上创建实际资源 (产生少量费用)
# GCS standard storage: ~$0.02/GB/月
# BigQuery: 按查询量计费，前 1TB/月免费
# Pub/Sub: 前 10GB/月免费

result = subprocess.run(
    f"cd /workspace/infrastructure/terraform && "
    f"terraform apply -var='project_id={PROJECT_ID}' -var='region={REGION}' -auto-approve 2>&1",
    shell=True, capture_output=True, text=True
)
# 打印最后 40 行
lines = result.stdout.strip().split('\n')
print('\n'.join(lines[-40:]))

Plan: 17 to add, 0 to change, 0 to destroy.
google_pubsub_topic.github_events_dlq: Creating...
google_pubsub_topic.github_events_raw: Creating...
google_bigquery_table.serving_daily_event_stats: Creating...
google_pubsub_topic.pipeline_status: Creating...
google_storage_bucket.temp: Creating...
google_bigquery_table.staging_github_events: Creating...
google_bigquery_table.raw_github_events: Creating...
google_storage_bucket.raw: Creating...
google_storage_bucket.staging: Creating...
google_bigquery_table.serving_top_repos: Creating...
google_bigquery_table.staging_github_events: Creation complete after 1s [id=projects/gcp-lab-data-engineering/datasets/staging/tables/github_events]
google_bigquery_table.serving_daily_event_stats: Creation complete after 1s [id=projects/gcp-lab-data-engineering/datasets/serving/tables/daily_event_stats]
google_bigquery_table.serving_top_repos: Creation complete after 1s [id=projects/gcp-lab-data-engineering/datasets/serving/tables/top_repos_daily]
google

In [8]:
# ===== 2.3 验证基础设施 =====
from google.cloud import storage, bigquery

# 验证 GCS buckets
gcs = storage.Client(project=PROJECT_ID)
print("=== GCS Buckets ===")
for b in gcs.list_buckets():
    if PROJECT_ID in b.name:
        print(f"  ✓ gs://{b.name}")

# 验证 BigQuery datasets + tables
bq = bigquery.Client(project=PROJECT_ID)
print("\n=== BigQuery ===")
for dataset_id in ["raw", "staging", "serving"]:
    tables = list(bq.list_tables(f"{PROJECT_ID}.{dataset_id}"))
    print(f"  {dataset_id}: {[t.table_id for t in tables]}")

=== GCS Buckets ===
  ✓ gs://gcp-lab-data-engineering-datalake-raw
  ✓ gs://gcp-lab-data-engineering-datalake-staging
  ✓ gs://gcp-lab-data-engineering-pipeline-temp

=== BigQuery ===
  raw: ['github_events']
  staging: ['github_events']
  serving: ['daily_event_stats', 'top_repos_daily']


---
# Part 3: 数据摄取层

## 设计原则
1. **幂等性**: Raw 层只追加，重跑产生新文件（通过时间戳区分）
2. **元数据注入**: 每条记录加 `_ingested_at` / `_source` / `_run_id`
3. **限速**: 内置令牌桶，遵守 GitHub API 限制
4. **Sink 抽象**: `local` (本地调试) / `gcs` (生产) 可切换

## 核心文件
- `ingestion/api_ingest.py` — GitHub Events 拉取 + 写入
- `ingestion/sinks/gcs_sink.py` — GCS 写入实现
- `ingestion/sinks/local_sink.py` — 本地写入（调试用）

## 文件命名规范
```
gs://{project}-datalake-raw/
  raw/github_events/
    dt=2024-01-15/
      events_20240115_020001.jsonl
      events_20240115_020045.jsonl   ← 重跑产生新文件，不覆盖
```

In [9]:
# ===== 3.1 本地测试摄取 (不需要 GCP) =====
import sys
sys.path.insert(0, '/workspace')

from ingestion.api_ingest import fetch_events

# 拉取少量数据做验证
from datetime import date
today = date.today().isoformat()

print(f"Fetching GitHub events for {today} (limit=5)...")
records = fetch_events(run_date=today, limit=5)
print(f"Fetched {len(records)} records")

if records:
    r = records[0]
    print(f"\n示例记录 (字段):")
    for k, v in r.items():
        val_str = str(v)[:60] + '...' if len(str(v)) > 60 else str(v)
        print(f"  {k}: {val_str}")

Fetching GitHub events for 2026-03-26 (limit=5)...
Fetched 5 records

示例记录 (字段):
  id: 9830863457
  type: PushEvent
  actor: {'id': 110542210, 'login': 'sossost', 'display_login': 'soss...
  repo: {'id': 1172228343, 'name': 'sossost/market-analyst', 'url': ...
  payload: {'repository_id': 1172228343, 'push_id': 32124188645, 'ref':...
  public: True
  created_at: 2026-03-26T05:01:28Z
  _ingested_at: 2026-03-26T05:06:29.425952


In [ ]:
# ===== 3.2 写入本地 (调试模式) =====
from ingestion.api_ingest import write_raw

write_raw(records, sink='local', run_date=today)

import os
local_dir = f"/workspace/data/raw/github_events/dt={today}"
if os.path.exists(local_dir):
    files = os.listdir(local_dir)
    print(f"写入到: {local_dir}")
    for f in files:
        size = os.path.getsize(f"{local_dir}/{f}")
        print(f"  {f} ({size:,} bytes)")

In [ ]:
# ===== 3.3 查看 GCS sink 实现 =====
# 文件: /workspace/ingestion/sinks/gcs_sink.py
with open('/workspace/ingestion/sinks/gcs_sink.py') as f:
    print(f.read())

In [ ]:
# ===== 3.4 写入 GCS (需要 GCP 权限) =====
from ingestion.api_ingest import write_raw

gcs_uri = write_raw(
    records,
    sink='gcs',
    run_date=today,
    bucket=RAW_BUCKET,
)
print(f"写入 GCS: {gcs_uri}")

# 验证
from google.cloud import storage
gcs_client = storage.Client(project=PROJECT_ID)
bucket = gcs_client.bucket(RAW_BUCKET)
blobs = list(bucket.list_blobs(prefix=f"raw/github_events/dt={today}/"))
print(f"GCS 文件数: {len(blobs)}")
for blob in blobs:
    print(f"  gs://{RAW_BUCKET}/{blob.name} ({blob.size:,} bytes)")

---
# Part 4: 数据质量框架

## 设计理念: SQL-based Assertions

```
每个 DQ 规则 = 一段 SQL
规则执行后:
  返回 0 行 → PASS ✓
  返回 N 行 → FAIL ✗ (这 N 行就是问题数据)

优势:
  - 直接在 BigQuery 上运行，无需拉数据到本地
  - SQL 版本化管理，DBA 可审阅
  - 失败时可直接查看具体问题记录
```

## 告警级别
| 级别 | 行为 |
|------|------|
| CRITICAL | 失败 → 抛出异常 → pipeline 停止，不写入下游 |
| WARNING | 失败 → 记录日志 → pipeline 继续执行 |

## 核心文件
- `data_quality/checks.py` — DQ 规则定义 + 执行引擎

In [ ]:
# ===== 4.1 查看 DQ 规则 =====
from data_quality.checks import RAW_CHECKS, STAGING_CHECKS, SERVING_CHECKS, Severity

all_checks = [
    ('raw',     RAW_CHECKS),
    ('staging', STAGING_CHECKS),
    ('serving', SERVING_CHECKS),
]

for layer, checks in all_checks:
    print(f"\n=== {layer} layer ({len(checks)} checks) ===")
    for c in checks:
        icon = '🔴' if c.severity == Severity.CRITICAL else '🟡'
        print(f"  {icon} {c.name}")
        print(f"     {c.description}")

In [ ]:
# ===== 4.2 运行 DQ 检查 (raw layer) =====
from data_quality.checks import run_checks, assert_no_critical_failures

print(f"Running DQ checks on raw layer for {today}...")
results = run_checks(PROJECT_ID, today, layer='raw')

print(f"\n{'Check Name':<40} {'Status':<8} {'Severity':<10} {'Failed Rows'}")
print('-' * 75)
for r in results:
    status = '✓ PASS' if r.passed else '✗ FAIL'
    print(f"{r.check_name:<40} {status:<8} {r.severity.value:<10} {r.failed_rows if not r.passed else ''}")

passed = sum(1 for r in results if r.passed)
print(f"\n总计: {passed}/{len(results)} 通过")

In [ ]:
# ===== 4.3 如何添加自定义规则 =====
from data_quality.checks import DQCheck, Severity

# 示例: 添加「PushEvent 占比不超过 80%」的规则
custom_check = DQCheck(
    name="staging_push_event_ratio",
    description="PushEvent 占所有事件不应超过 80% (异常数据源)",
    severity=Severity.WARNING,
    sql="""
        SELECT
          COUNTIF(event_type = 'PushEvent') / COUNT(*) AS push_ratio
        FROM `{project}.staging.github_events`
        WHERE event_date = '{run_date}'
        HAVING push_ratio > 0.8
    """
)

# 添加到 STAGING_CHECKS 列表即可纳入自动检测
# from data_quality.checks import STAGING_CHECKS
# STAGING_CHECKS.append(custom_check)

print(f"自定义规则: {custom_check.name}")
print(f"SQL 模板:\n{custom_check.sql}")

---
# Part 5: ETL 转换层

## 转换逻辑

### Step 3: GCS → BigQuery raw (Load Job)
```
文件: etl/load.py
策略: BigQuery Load Job (WRITE_APPEND)
      每次 load 只加载当日分区文件
      重跑保护: 先 DELETE 当日分区再 load
```

### Step 4a: raw → staging (SQL MERGE)
```sql
MERGE staging.github_events T
USING (
  SELECT id, type, DATE(created_at), actor.login, repo.name, ...
  FROM raw.github_events
  WHERE DATE(_ingested_at) = '{run_date}'
) S
ON T.event_id = S.event_id AND T.event_date = S.event_date
WHEN NOT MATCHED THEN INSERT ...  -- 幂等: 已存在不重复插入
```

### Step 4b: staging → serving (SQL MERGE)
```sql
-- daily_event_stats: 按 event_date + event_type 聚合
-- top_repos_daily:   按 event_date + repo_name 聚合 TOP 100
MERGE serving.daily_event_stats
WHEN MATCHED THEN UPDATE ...  -- 重跑: 更新聚合值
WHEN NOT MATCHED THEN INSERT ...
```

In [ ]:
# ===== 5.1 Step 3: Load GCS → BigQuery raw =====
from etl.load import load_gcs_to_bq

print(f"Loading GCS → BigQuery raw for {today}...")
rows_loaded = load_gcs_to_bq(
    project_id=PROJECT_ID,
    run_date=today,
    source_bucket=RAW_BUCKET,
)
print(f"Loaded: {rows_loaded} rows")

In [ ]:
# ===== 5.2 验证 raw 层数据 =====
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)

query = f"""
SELECT
  DATE(_ingested_at) AS ingested_date,
  COUNT(*)           AS total_events,
  COUNT(DISTINCT type) AS distinct_types,
  MIN(created_at)    AS earliest,
  MAX(created_at)    AS latest
FROM `{PROJECT_ID}.raw.github_events`
WHERE DATE(_ingested_at) = '{today}'
GROUP BY 1
"""

for row in bq.query(query):
    print(f"ingested_date:   {row['ingested_date']}")
    print(f"total_events:    {row['total_events']:,}")
    print(f"distinct_types:  {row['distinct_types']}")
    print(f"earliest:        {row['earliest']}")
    print(f"latest:          {row['latest']}")

In [ ]:
# ===== 5.3 Step 4: Transform raw → staging → serving =====
from etl.transform import run_transforms

print(f"Running transforms for {today}...")
stats = run_transforms(PROJECT_ID, today)

print("\n=== Transform Results ===")
for step, rows in stats.items():
    print(f"  {step}: {rows} rows affected")

In [ ]:
# ===== 5.4 查询 serving 层结果 =====
# 按 event_type 查看今日统计
query = f"""
SELECT
  event_type,
  event_count,
  unique_actors,
  unique_repos
FROM `{PROJECT_ID}.serving.daily_event_stats`
WHERE event_date = '{today}'
ORDER BY event_count DESC
LIMIT 10
"""

print(f"\n=== Daily Event Stats ({today}) ===")
print(f"{'event_type':<35} {'count':>8} {'actors':>8} {'repos':>8}")
print('-' * 62)
for row in bq.query(query):
    print(f"{row['event_type']:<35} {row['event_count']:>8,} {row['unique_actors']:>8,} {row['unique_repos']:>8,}")

In [ ]:
# ===== 5.5 Top Repos 今日排行 =====
query = f"""
SELECT repo_name, event_count, unique_actors, event_types
FROM `{PROJECT_ID}.serving.top_repos_daily`
WHERE event_date = '{today}'
ORDER BY event_count DESC
LIMIT 10
"""

print(f"\n=== Top 10 Repos ({today}) ===")
for i, row in enumerate(bq.query(query), 1):
    print(f"  {i:>2}. {row['repo_name']:<45} events={row['event_count']:>5,}")

---
# Part 6: 编排与调度

## 两种调度方案

### 方案 A: Cloud Scheduler + Cloud Run (推荐，轻量)
```
Cloud Scheduler (每日 02:00)
  → HTTP POST → Cloud Run (容器化 pipeline)
  → 完整执行 orchestration/pipeline.py

优点: 无需维护 Airflow，成本低，弹性伸缩
适用: 单链路 pipeline，任务依赖简单
```

### 方案 B: Cloud Composer (托管 Airflow)
```
Cloud Composer (Airflow)
  → DAG: github_events_daily
  → 支持复杂依赖、重试、SLA 监控、回溯

优点: 功能完整，可视化 DAG，支持数十个任务编排
适用: 多 pipeline 协调，复杂依赖，跨团队 SLA
成本: ~$300/月起 (Composer 2 Small)
```

In [ ]:
# ===== 6.1 手动执行完整 Pipeline (方案 A: 脚本模式) =====
from orchestration.pipeline import run_pipeline

print(f"=== Running full pipeline for {today} ===")
summary = run_pipeline(
    run_date=today,
    project_id=PROJECT_ID,
    raw_bucket=RAW_BUCKET,
    event_limit=100,      # 测试用: 限制 100 条
)

print("\n=== Pipeline Summary ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
# ===== 6.2 方案 B: Airflow DAG 模板 =====
# 如果使用 Cloud Composer，将以下 DAG 文件上传到 Composer 的 DAGs bucket

AIRFLOW_DAG = '''
# dags/github_events_daily.py

from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.empty import EmptyOperator

PROJECT_ID  = "gcp-lab-data-engineering"
RAW_BUCKET  = f"{PROJECT_ID}-datalake-raw"

DEFAULT_ARGS = {
    "owner": "data-engineering",
    "retries": 3,
    "retry_delay": timedelta(minutes=5),
    "retry_exponential_backoff": True,
    "email_on_failure": True,
    "email": ["oncall@company.com"],
    "execution_timeout": timedelta(hours=2),
}

with DAG(
    dag_id="github_events_daily",
    default_args=DEFAULT_ARGS,
    schedule_interval="0 2 * * *",   # 每天 02:00 UTC
    start_date=datetime(2024, 1, 1),
    catchup=False,
    max_active_runs=1,
    tags=["github", "daily"],
) as dag:

    start = EmptyOperator(task_id="start")

    ingest = PythonOperator(
        task_id="ingest_github_events",
        python_callable=lambda **ctx: __import__("ingestion.api_ingest", fromlist=["fetch_events", "write_raw"]),
        op_kwargs={"run_date": "{{ ds }}", "bucket": RAW_BUCKET},
        sla=timedelta(hours=1),
    )

    dq_raw = PythonOperator(
        task_id="dq_raw",
        python_callable=lambda **ctx: __import__("data_quality.checks", fromlist=["run_checks"]),
        op_kwargs={"project_id": PROJECT_ID, "run_date": "{{ ds }}", "layer": "raw"},
    )

    load_bq = PythonOperator(task_id="load_gcs_to_bq",    python_callable=lambda **ctx: None, op_kwargs={"run_date": "{{ ds }}"})
    transform = PythonOperator(task_id="transform",        python_callable=lambda **ctx: None, op_kwargs={"run_date": "{{ ds }}"})
    dq_staging = PythonOperator(task_id="dq_staging",     python_callable=lambda **ctx: None, op_kwargs={"run_date": "{{ ds }}"})
    dq_serving = PythonOperator(task_id="dq_serving",     python_callable=lambda **ctx: None, op_kwargs={"run_date": "{{ ds }}"})
    end = EmptyOperator(task_id="end")

    start >> ingest >> dq_raw >> load_bq >> transform >> [dq_staging, dq_serving] >> end
'''

print("DAG 依赖链:")
print("""
start
  └─► ingest (SLA: 1h)
        └─► dq_raw
              └─► load_bq
                    └─► transform
                          ├─► dq_staging ─┐
                          └─► dq_serving  ─► end
""")

In [ ]:
# ===== 6.3 Cloud Scheduler 设置 (命令行) =====
SCHEDULER_CMD = f"""
# 部署 Cloud Scheduler 每日触发 pipeline
# 先把 pipeline 打包为 Cloud Run 服务，然后:

gcloud scheduler jobs create http github-events-daily-pipeline \\
  --location={REGION} \\
  --schedule='0 2 * * *' \\
  --time-zone='UTC' \\
  --uri='https://github-events-pipeline-xxxx-uc.a.run.app/run' \\
  --http-method=POST \\
  --message-body='{{"date": "today"}}' \\
  --oidc-service-account-email=svc-ingest@{PROJECT_ID}.iam.gserviceaccount.com

# 手动触发测试:
gcloud scheduler jobs run github-events-daily-pipeline --location={REGION}
"""
print(SCHEDULER_CMD)

---
# Part 7: 监控与可观测性

## GCP 监控体系

```
Cloud Logging  → 结构化日志 (自动从 stdout JSON 捕获)
Cloud Monitoring → 自定义指标 + 内置 BigQuery/GCS 指标
Cloud Alerting → 基于指标和日志的告警规则
Pub/Sub → pipeline 状态事件
```

## 关键告警规则
| 告警 | 条件 | 通知 |
|------|------|------|
| Pipeline 失败 | ERROR 日志出现 | PagerDuty / Slack |
| 数据量异常 | 今日 vs 昨日偏差 > 30% | Slack |
| DQ 失败 | CRITICAL check 失败 | PagerDuty |
| 数据延迟 | 08:00 前 serving 层无数据 | PagerDuty |
| GCS 存储超限 | bucket 大小 > 500GB | Slack |

In [ ]:
# ===== 7.1 结构化日志 (Cloud Logging 自动捕获 JSON stdout) =====
import json
from datetime import datetime, timezone

def structured_log(severity: str, message: str, **fields):
    """
    输出符合 Cloud Logging 结构化日志格式的 JSON。
    在 Cloud Run / Compute Engine 上运行时，Cloud Logging 自动解析。
    """
    record = {
        "severity": severity,                          # Cloud Logging 识别此字段
        "message": message,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        **fields,
    }
    print(json.dumps(record))

# 使用示例
structured_log(
    "INFO",
    "Pipeline step completed",
    pipeline="github_events_daily",
    step="ingest",
    run_date=today,
    rows=1500,
    duration_s=12.3,
)

In [ ]:
# ===== 7.2 自定义指标写入 Cloud Monitoring =====
# 安装: pip install google-cloud-monitoring

MONITORING_CODE = """
from google.cloud import monitoring_v3
from google.protobuf import timestamp_pb2
import time

def write_pipeline_metric(project_id: str, metric_name: str, value: float, labels: dict):
    client = monitoring_v3.MetricServiceClient()
    project_name = f'projects/{project_id}'

    series = monitoring_v3.TimeSeries()
    series.metric.type = f'custom.googleapis.com/pipeline/{metric_name}'
    series.metric.labels.update(labels)
    series.resource.type = 'global'

    now = time.time()
    point = monitoring_v3.Point()
    point.value.double_value = value
    point.interval.end_time.seconds = int(now)
    series.points.append(point)

    client.create_time_series(name=project_name, time_series=[series])

# 使用
write_pipeline_metric(
    project_id='gcp-lab-data-engineering',
    metric_name='ingested_rows',
    value=1500.0,
    labels={'pipeline': 'github_events_daily', 'layer': 'raw'}
)
"""
print("Cloud Monitoring 自定义指标代码:")
print(MONITORING_CODE)

In [ ]:
# ===== 7.3 数据量异常检测 (查询 BigQuery 历史数据) =====
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)

anomaly_query = f"""
WITH daily_counts AS (
  SELECT
    DATE(_ingested_at) AS dt,
    COUNT(*)           AS row_count
  FROM `{PROJECT_ID}.raw.github_events`
  WHERE DATE(_ingested_at) >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY)
  GROUP BY 1
),
with_stats AS (
  SELECT
    dt,
    row_count,
    AVG(row_count) OVER (ORDER BY dt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) AS moving_avg_7d,
    STDDEV(row_count) OVER (ORDER BY dt ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING) AS stddev_7d
  FROM daily_counts
)
SELECT
  dt,
  row_count,
  ROUND(moving_avg_7d) AS avg_7d,
  CASE
    WHEN moving_avg_7d IS NULL THEN 'INSUFFICIENT_HISTORY'
    WHEN ABS(row_count - moving_avg_7d) > 2 * COALESCE(stddev_7d, moving_avg_7d * 0.3)
    THEN '⚠ ANOMALY'
    ELSE 'OK'
  END AS status
FROM with_stats
ORDER BY dt DESC
"""

print("=== 数据量异常检测 (近 7 天) ===")
print(f"{'Date':<14} {'Count':>8} {'7d Avg':>8} {'Status'}")
print('-' * 45)
for row in bq.query(anomaly_query):
    avg = f"{row['avg_7d']:,.0f}" if row['avg_7d'] else 'N/A'
    print(f"{str(row['dt']):<14} {row['row_count']:>8,} {avg:>8} {row['status']}")

---
# Part 8: 安全与数据治理

## GCP 安全架构

### Service Account 权限矩阵
| SA | GCS Raw | GCS Staging | BQ Raw | BQ Staging | BQ Serving |
|----|---------|-------------|--------|------------|------------|
| `svc-ingest` | Write | — | — | — | — |
| `svc-etl` | Read | Write | Write | Write | Write |
| `svc-analytics` | — | — | — | — | Read |

*(已通过 `infrastructure/terraform/iam.tf` 和 `gcs.tf` 配置)*

### Secret Manager
```
项目中需要保密的配置:
  - API Keys (如需要 GitHub token 提高速率限制)
  - 数据库连接串
  - Slack Webhook URL (告警通知)

绝不能做:
  - ❌ 在代码中硬编码 API Key
  - ❌ 将 key.json 提交到 Git
  - ❌ 在日志中打印 secret 内容
```

### BigQuery 列级安全 (Column-level Security)
```sql
-- 对 actor_login 字段打上 SENSITIVE 标签
-- analytics SA 访问时自动脱敏
CREATE OR REPLACE TABLE staging.github_events
OPTIONS (description = 'Cleaned GitHub events')
AS SELECT * FROM ...

-- 在 BigQuery Policy Tags 中设置:
-- actor_login → policy_tag: SENSITIVE/PII
-- 未授权 SA 查询时返回 NULL 或报错
```

In [ ]:
# ===== 8.1 从 Secret Manager 读取密钥 (模式示范) =====

def get_secret(project_id: str, secret_name: str, version: str = "latest") -> str:
    """
    从 GCP Secret Manager 安全读取密钥。
    在 Cloud Run/Compute Engine 上会自动使用 SA 的权限。
    """
    from google.cloud import secretmanager

    client = secretmanager.SecretManagerServiceClient()
    name   = f"projects/{project_id}/secrets/{secret_name}/versions/{version}"
    resp   = client.access_secret_version(request={"name": name})
    return resp.payload.data.decode("UTF-8")

# 使用示例 (取消注释后可运行):
# github_token = get_secret(PROJECT_ID, "github-api-token")
# slack_webhook = get_secret(PROJECT_ID, "slack-webhook-oncall")

print("Secret 使用规范:")
print("  1. 创建: gcloud secrets create github-api-token --data-file=token.txt")
print("  2. 授权: gcloud secrets add-iam-policy-binding github-api-token")
print("           --member=serviceAccount:svc-ingest@... --role=roles/secretmanager.secretAccessor")
print("  3. 代码中: secret = get_secret(PROJECT_ID, 'github-api-token')")
print("  ✓ 不要 print(secret), 不要写入日志")

In [ ]:
# ===== 8.2 数据血缘 (BigQuery INFORMATION_SCHEMA) =====

lineage_query = f"""
-- 查询 staging 表的来源 (BigQuery 自动记录)
SELECT
  destination_table.project_id,
  destination_table.dataset_id,
  destination_table.table_id,
  job_type,
  creation_time,
  user_email
FROM `{PROJECT_ID}`.`region-us`.INFORMATION_SCHEMA.JOBS
WHERE destination_table.dataset_id = 'staging'
  AND destination_table.table_id   = 'github_events'
  AND DATE(creation_time) = '{today}'
ORDER BY creation_time DESC
LIMIT 5
"""

print("查询 BQ 作业历史 (数据血缘) ...")
try:
    for row in bq.query(lineage_query):
        print(f"  {row['creation_time']} | {row['job_type']} | {row['destination_table']['dataset_id']}.{row['destination_table']['table_id']}")
except Exception as e:
    print(f"  (需要 bigquery.jobs.list 权限) {e}")

---
# Part 9: CI/CD

## Cloud Build 配置

```yaml
# cloudbuild.yaml
steps:
  # 1. 安装依赖
  - name: python:3.11
    entrypoint: pip
    args: [install, -r, requirements.txt, -r, requirements-dev.txt]

  # 2. 单元测试
  - name: python:3.11
    entrypoint: python
    args: [-m, pytest, tests/, -v, --tb=short]

  # 3. 数据质量规则语法校验
  - name: python:3.11
    entrypoint: python
    args: [-c, from data_quality.checks import STAGING_CHECKS; print(len(STAGING_CHECKS), 'checks OK')]

  # 4. Terraform plan (Review only)
  - name: hashicorp/terraform:1.6
    dir: infrastructure/terraform
    args: [plan, -var=project_id=$PROJECT_ID]

  # 5. 仅 main 分支才 apply + 部署
  - name: python:3.11
    entrypoint: bash
    args:
      - -c
      - |
        if [ "$BRANCH_NAME" = "main" ]; then
          terraform apply -auto-approve
          gcloud run deploy github-events-pipeline --source=.
        fi

options:
  logging: CLOUD_LOGGING_ONLY
```

## 部署流程
```
PR → Cloud Build 自动跑测试 + tf plan
  → Code Review 通过
  → Merge to main
  → Cloud Build apply + 部署 Cloud Run
```

## 数据回滚
```sql
-- BigQuery 时间旅行回滚 (7 天内)
-- 错误写入后，恢复到 1 小时前的状态:
CREATE OR REPLACE TABLE staging.github_events AS
SELECT * FROM staging.github_events
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR);

-- 或用 snapshot decorator:
SELECT * FROM `project.staging.github_events@-3600000`  -- 1小时前 (毫秒)
```

---
# Part 10: 实施检查清单

## 上线前 Checklist

### 基础设施
- [ ] `terraform apply` 成功，所有资源已创建
- [ ] GCS bucket 权限验证: `gsutil ls gs://{project}-datalake-raw/`
- [ ] BigQuery dataset 权限验证: `bq ls {project}:raw`
- [ ] Service account 可以正常写入 GCS
- [ ] Pub/Sub topic 已创建并测试

### 摄取层
- [ ] 本地 `--sink local` 测试通过
- [ ] GCS sink 写入验证 (`gsutil cat gs://.../*.jsonl | head`)
- [ ] 元数据字段 (`_ingested_at`, `_run_id`) 已注入
- [ ] 重跑测试: 相同日期运行 3 次，GCS 产生 3 个不同文件（不覆盖）

### ETL 层
- [ ] BQ Load Job 成功，row count 与 GCS 文件行数一致
- [ ] MERGE 幂等性验证: 重跑 staging transform，行数不增加
- [ ] serving 层数据可正常查询
- [ ] BigQuery 分区和聚簇字段正确

### 数据质量
- [ ] 所有 CRITICAL 规则在正常数据下通过
- [ ] 故意注入坏数据，验证 CRITICAL 规则能捕获并阻断
- [ ] WARNING 规则失败时不阻断 pipeline

### 监控
- [ ] Cloud Logging 中能看到结构化日志
- [ ] 告警规则已配置并测试通知渠道
- [ ] 数据量异常检测 SQL 已验证

### 安全
- [ ] 无明文密钥在代码中
- [ ] `.gitignore` 包含 `*.json`, `terraform.tfvars`, `.env`
- [ ] `terraform.tfstate` 不包含 secret（或已加密存储）

### 上线后 72 小时
- [ ] T+1: pipeline 首次自动运行成功
- [ ] T+1: serving 层数据量与预期一致
- [ ] T+2: 告警规则无误报
- [ ] T+3: 完成首次手动回溯测试

In [ ]:
# ===== 10.1 一键健康检查 =====
from datetime import date
from google.cloud import storage, bigquery

def health_check(project_id: str, run_date: str) -> dict:
    """快速验证当日 pipeline 执行状态"""
    results = {}
    bq = bigquery.Client(project=project_id)
    gcs = storage.Client(project=project_id)

    # GCS raw 文件
    raw_bucket = f"{project_id}-datalake-raw"
    try:
        blobs = list(gcs.bucket(raw_bucket).list_blobs(prefix=f"raw/github_events/dt={run_date}/"))
        results["gcs_raw_files"] = len(blobs)
        results["gcs_raw_status"] = "✓" if blobs else "✗ no files"
    except Exception as e:
        results["gcs_raw_status"] = f"✗ {e}"

    # BQ 各层行数
    for layer, table, date_col in [
        ("raw",     "github_events",     "_ingested_at"),
        ("staging", "github_events",     "event_date"),
        ("serving", "daily_event_stats", "event_date"),
    ]:
        try:
            q = f"SELECT COUNT(*) AS n FROM `{project_id}.{layer}.{table}` WHERE DATE({date_col}) = '{run_date}'"
            rows = list(bq.query(q))[0]["n"]
            results[f"bq_{layer}_rows"] = rows
            results[f"bq_{layer}_status"] = "✓" if rows > 0 else "✗ 0 rows"
        except Exception as e:
            results[f"bq_{layer}_status"] = f"✗ {e}"

    return results


print(f"=== Pipeline Health Check: {today} ===")
status = health_check(PROJECT_ID, today)
for k, v in status.items():
    print(f"  {k:<28}: {v}")

In [ ]:
# ===== 10.2 快速参考: 常用命令 =====

CHEATSHEET = f"""
=== 日常操作速查 ===

# 运行今日 pipeline
make run DATE=$(date +%Y-%m-%d)

# 仅摄取 (本地调试)
make ingest-local DATE=2024-01-15

# 单独运行 DQ 检查
make dq-raw DATE=2024-01-15
make dq-staging DATE=2024-01-15

# 回溯历史数据 (重跑指定日期)
for d in 2024-01-10 2024-01-11 2024-01-12; do make run DATE=$d; done

# 查看最新 raw 数据
bq query --nouse_legacy_sql \
  'SELECT id, type, created_at FROM `{PROJECT_ID}.raw.github_events` LIMIT 10'

# 查看今日 serving 统计
bq query --nouse_legacy_sql \
  "SELECT * FROM `{PROJECT_ID}.serving.daily_event_stats` WHERE event_date = CURRENT_DATE()"

# 列出 GCS 原始文件
gsutil ls gs://{PROJECT_ID}-datalake-raw/raw/github_events/dt=$(date +%Y-%m-%d)/

# 查看 BQ 表大小
bq show --format=prettyjson {PROJECT_ID}:raw.github_events | grep -E 'numBytes|numRows'

# BigQuery 数据回滚 (7天内)
bq query --nouse_legacy_sql \
  'SELECT COUNT(*) FROM `{PROJECT_ID}.staging.github_events` FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)'
"""
print(CHEATSHEET)